# convT-kernel-axis-swap — faded example 3: Complete the swap that loads a Conv2d weight into ConvTranspose2d

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `convT-kernel-axis-swap`. The last cell reports your progress on the `CNN: ConvT kernel axis swap` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT kernel axis swap` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`convT-kernel-axis-swap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "convT-kernel-axis-swap"
DD_SUBTOPIC = "CNN: ConvT kernel axis swap"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

To reuse a `Conv2d` weight `(OC, IC, KH, KW)` inside a `nn.ConvTranspose2d`, you must first reorder it into transposed-conv layout `(IC, OC, KH, KW)` by swapping the two channel axes. The result must be contiguous so `weight.data.copy_` accepts it; spatial axes are never touched.

## Faded exercise 3

### Faded — conv weight into a ConvTranspose2d

Implement `conv_to_convT_weight(conv_weight)`. Given a Conv2d weight `(OC, IC, KH, KW)`, return the ConvT2d weight `(IC, OC, KH, KW)` (channel axes swapped, spatial axes unchanged, contiguous). Complete the blanked line that performs the swap.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def conv_to_convT_weight(conv_weight: Tensor) -> Tensor:
    convT_weight = rearrange(conv_weight, 'o i kh kw -> i o kh kw').contiguous()
    return convT_weight


def _test():
    t.manual_seed(0)
    oc, ic, kh, kw = 5, 3, 4, 4
    conv_w = t.randn(oc, ic, kh, kw)        # Conv2d layout (OC, IC, KH, KW)
    convT_w = conv_to_convT_weight(conv_w)
    assert tuple(convT_w.shape) == (ic, oc, kh, kw), tuple(convT_w.shape)
    assert convT_w.is_contiguous(), 'result must be contiguous'
    assert t.allclose(convT_w, conv_w.transpose(0, 1)), 'channel axes not swapped correctly'
    # must actually load into a ConvTranspose2d with matching in/out channels
    convT = t.nn.ConvTranspose2d(ic, oc, kernel_size=(kh, kw), bias=False)
    convT.weight.data.copy_(convT_w)
    assert t.allclose(convT.weight.data, conv_w.transpose(0, 1))


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def conv_to_convT_weight(conv_weight: Tensor) -> Tensor:
    convT_weight = rearrange(conv_weight, 'o i kh kw -> i o kh kw').contiguous()
    return convT_weight
```
</details>